<div dir="rtl" lang="he" align="right" markdown="1">

# הנדסת הקשר

שיחה עם סוכן מתארכת. ההיסטוריה גדלה, העלות עולה, והאינסטינקט הראשון הוא לסכם את ההתחלה.

הפרק הזה מודד מה זה באמת עולה, והתוצאה הפוכה מהציפייה: **שמירה על הכל יצאה זולה יותר מסיכום.** לא בגלל שהסיכום לא מקצר — הוא מקצר מצוין — אלא בגלל ה-`cache`.

החלק הטוב בפרק הוא שאת ההסבר אפשר להכריע בלי לקרוא לאף מודל. זו השוואת בתים.

</div>

In [ ]:
import json
from pathlib import Path

from aihe import viz
from aihe.context import (
    cache_hit_ratio,
    capped_history,
    estimate_cost,
    planted_fact_probe,
    prefix_is_stable,
    render,
    summarise_old,
)
from aihe.models import asker

HERE = Path("data") if Path("data/probes.json").exists() else Path("chapters/04-context/data")
CASSETTES = HERE.parent / "cassettes"
plan = json.loads((HERE / "probes.json").read_text(encoding="utf-8"))
print(len(plan["probes"]), "probes, cap =", plan["cap"], "characters")

<div dir="rtl" lang="he" align="right" markdown="1">

## הניסוי

לכל בדיקה יש אותו מבנה. בתחילת השיחה נשתלת עובדה שאי אפשר לנחש — `TANGERINE-7741` — ואז נערמים עליה חמישה תורות של פלט כלים אמיתי למראה, ובסוף נשאלת שאלה שרק העובדה עונה עליה.

זה חשוב: עובדה שאפשר לנחש הייתה מודדת את הידע הכללי של המודל. עובדה שרירותית מודדת רק מה שההקשר שמר.

</div>

In [ ]:
probe = plan["probes"][0]
turns = planted_fact_probe(probe["fact"], probe["question"], probe["filler"])
print("fact    :", probe["fact"])
print("question:", probe["question"])
print("turns   :", len(turns), "| roles:", sorted({t.role for t in turns}))
print("\nrendered:", len(render(turns)), "characters")

<div dir="rtl" lang="he" align="right" markdown="1">

## שלוש אסטרטגיות

הראשונה היא לא לעשות כלום — מוסיפים תורות ולא נוגעים במה שכבר נשלח.

השנייה היא לסכם: מחליפים את כל מה שקדם בתקציר קצר ומשאירים את התורות האחרונים. זה מה שרוב המערכות עושות.

השלישית היא לחתוך את פלט הכלים, וההבדל שלה מהשנייה עדין וחשוב: החיתוך קורה **בזמן הכתיבה**. הפלט הארוך מעולם לא נכנס להיסטוריה, ולכן שום טקסט שכבר נשלח לא נכתב מחדש.

</div>

In [ ]:
variants = {
    "keep_all": turns,
    "summarised": summarise_old(turns, keep_recent=4, summary=plan["summary"]),
    "capped": capped_history(turns, limit=plan["cap"]),
}
base = len(render(turns))
for name, history in variants.items():
    size = len(render(history))
    kept = probe["fact"] in render(history)
    print(f"  {name:12s} {size:5d} chars  saved {1 - size/base:4.0%}  fact still present: {kept}")

<div dir="rtl" lang="he" align="right" markdown="1">

## המנגנון, בהשוואת בתים

וכאן ההסבר. ה-`cache` של ספק מודלים פוגע רק על `prefix` **שלא השתנה**. כל עוד מוסיפים טקסט לסוף, כל מה שקדם עדיין שמור ומשולם בזול.

ברגע שמסכמים, תחילת השיחה היא טקסט אחר. אין `prefix` משותף, ולכן אין `cache` — והתור הבא משלם מחיר מלא על כל ההיסטוריה.

התא הבא מכריע את זה בלי מודל, בלי רשת ובלי עלות. זו פונקציה של שתי מחרוזות.

</div>

In [ ]:
before = render(turns[:-1])          # the conversation one turn ago

for name, history in variants.items():
    after = render(history)
    print(f"  {name:12s} prefix still valid: {str(prefix_is_stable(before, after)):5s}  "
          f"cache kept: {cache_hit_ratio(before, after):4.0%}")

In [ ]:
# what that does to the bill for the next turn
for name, history in variants.items():
    after = render(history)
    cached = before if prefix_is_stable(before, after) else ""
    print(f"  {name:12s} cost {estimate_cost(after, cached_prefix=cached):.6f}  "
          f"(cold would be {estimate_cost(after):.6f})")

<div dir="rtl" lang="he" align="right" markdown="1">

## ומה עם הזיכרון עצמו

חיסכון הוא חצי מהסיפור. החצי השני הוא האם המודל עדיין יודע את התשובה.

נריץ את כל שש הבדיקות תחת שלוש האסטרטגיות. התשובות הוקלטו מראש, כך שהתא רץ בלי אינטרנט ובלי עלות.

</div>

In [ ]:
import re

ask = asker(cassettes=CASSETTES, model="Llama-3.2-3B-Instruct-Q4_K_M",
            temperature=0.0, max_tokens=64, seed=7)

recalled = {name: 0 for name in variants}
sizes = {name: 0 for name in variants}
for item in plan["probes"]:
    t = planted_fact_probe(item["fact"], item["question"], item["filler"])
    wanted = re.search(r"is ([A-Za-z0-9#\-]+(?: [A-Za-z]+)?)\.", item["fact"]).group(1).lower()
    for name, history in (("keep_all", t),
                          ("summarised", summarise_old(t, 4, plan["summary"])),
                          ("capped", capped_history(t, plan["cap"]))):
        sizes[name] += len(render(history))
        recalled[name] += wanted in ask(render(history)).lower()

In [ ]:
total = sizes["keep_all"]
print(f"{'strategy':12s} {'chars':>7s} {'saved':>7s} {'fact recalled':>14s}")
for name in ("keep_all", "summarised", "capped"):
    print(f"{name:12s} {sizes[name]:7d} {1 - sizes[name]/total:7.0%} "
          f"{recalled[name]}/{len(plan['probes']):>12}")

viz.climb({k: recalled[k] / len(plan["probes"]) for k in ("summarised", "capped", "keep_all")},
          label="facts recalled", title="what each context strategy remembered")

<div dir="rtl" lang="he" align="right" markdown="1">

## מה לוקחים מכאן

**סיכום הוא מוצא אחרון, לא ראשון.** הוא חסך הכי הרבה תווים והיה היחיד שאיבד את העובדה — בכל שש הבדיקות, בלי יוצא מן הכלל.

**חיתוך בזמן כתיבה הוא הניצחון הזול.** הוא חסך כמעט חצי מהתווים ולא עלה כלום בשליפה, כי הוא לא כותב מחדש שום דבר שכבר נשלח.

**הבדיקה הכי חשובה כאן לא דורשת מודל.** האם ה-`prefix` הקודם עדיין תקף — זו שאלה על שתי מחרוזות, והיא נכנסת לבדיקות אוטומטיות בלי רשת ובלי עלות. אם משנים אסטרטגיית הקשר, זו הבדיקה שתתפוס את הרגרסיה.

**והסתייגות על המספרים.** שמירת הכל שלפה את העובדה ב-4 מתוך 6 ולא ב-6 מתוך 6. שתי ההחמצות הן מודל בגודל `3B` שלא מוצא עובדה שנמצאת מולו — מגבלה של המודל, לא של האסטרטגיה. מה שהפרק נשען עליו הוא ההשוואה בין האסטרטגיות על אותן בדיקות בדיוק, ושם ההפרש מוחלט: ארבע מול אפס.

</div>

In [ ]:
print(f"summarised: saved {1 - sizes['summarised']/total:.0%} of the characters, "
      f"recalled {recalled['summarised']}/{len(plan['probes'])}")
print(f"capped    : saved {1 - sizes['capped']/total:.0%} of the characters, "
      f"recalled {recalled['capped']}/{len(plan['probes'])}")